In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/transformer/raw_transformer_data.parquet


# 0 — Imports & Global Settings

In [2]:
# Core
import numpy as np
import pandas as pd
import math
import os

# ML / Torch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Sklearn
from sklearn.preprocessing import StandardScaler

# Utils
import joblib
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  # add for GPU reproducibility

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


# 1 — Load Preprocessed Data

In [3]:
DATA_PATH = "/kaggle/input/transformer"
df = pd.read_parquet(DATA_PATH)

print("Profiles:", df["profile_id"].nunique())
print("Depth points:", len(df))

assert "Depth_raw" in df.columns
assert "profile_id" in df.columns

Profiles: 2201
Depth points: 444417


# 2 — Feature / Target Definition

In [4]:
FEATURE_COLS = [
    "LON", "LAT", "Depth(m)", "TEMP(degC)",
    "time_days", "doy_sin", "doy_cos"
]

TARGET_COL = "PSAL(psu)"

# 3 — Outer TEST Split (Profile-wise)

In [5]:
def trainval_test_split_profiles(df, test_frac=0.15, seed=42):
    profiles = df["profile_id"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(profiles)

    n_test = int(len(profiles) * test_frac)
    return profiles[n_test:], profiles[:n_test]

trainval_profiles, test_profiles = trainval_test_split_profiles(df)

df_trainval = df[df.profile_id.isin(trainval_profiles)].copy()
df_test     = df[df.profile_id.isin(test_profiles)].copy()

print("TrainVal profiles:", df_trainval.profile_id.nunique())
print("Test profiles:", df_test.profile_id.nunique())

# DEPTH NORMALIZATION
MAX_DEPTH = df["Depth_raw"].max()
df_trainval["Depth(m)"] = df_trainval["Depth(m)"] / MAX_DEPTH
df_test["Depth(m)"]     = df_test["Depth(m)"] / MAX_DEPTH

TrainVal profiles: 1871
Test profiles: 330


# 4 — Define Spatio-Temporal Blocks (TRAINVAL ONLY)

In [6]:
# ============================================================
# CELL 4 — DEFINE SPATIO-TEMPORAL BLOCKS (PROFILE LEVEL)
# ============================================================

import numpy as np
import pandas as pd


def define_spatial_blocks(trainval_df):
    """
    Returns a function that maps profile_id -> spatial_block
    Block assignment is based on PROFILE-LEVEL mean LAT/LON
    """

    # ---- profile-level coordinates ----
    profile_coords = (
        trainval_df
        .groupby("profile_id")
        .agg(
            LAT=("LAT", "mean"),
            LON=("LON", "mean")
        )
    )

    lat_mid = profile_coords["LAT"].median()
    lon_mid = profile_coords["LON"].median()

    # ---- assign block per profile ----
    spatial_map = (
        (profile_coords["LAT"] > lat_mid).astype(int) * 2 +
        (profile_coords["LON"] > lon_mid).astype(int)
    ).to_dict()

    def fn(profile_id):
        return spatial_map[profile_id]

    return fn


def define_temporal_blocks(trainval_df, n_blocks=10):
    """
    Returns a function that maps profile_id -> temporal_block
    Block assignment is based on PROFILE-LEVEL mean time_days
    """

    profile_time = (
        trainval_df
        .groupby("profile_id")
        .agg(time_days=("time_days", "mean"))
    )

    qs = np.quantile(
        profile_time["time_days"],
        np.linspace(0, 1, n_blocks + 1)
    )

    temporal_blocks = (
    np.searchsorted(qs, profile_time["time_days"], side="right") - 1
    )

    temporal_blocks = np.clip(temporal_blocks, 0, n_blocks - 1)

    temporal_map = pd.Series(
        temporal_blocks,
        index=profile_time.index
    ).to_dict()


    def fn(profile_id):
        return temporal_map[profile_id]

    return fn


# ------------------------------------------------------------
# APPLY BLOCK FUNCTIONS (TRAINVAL ONLY)
# ------------------------------------------------------------

spatial_fn  = define_spatial_blocks(df_trainval)
temporal_fn = define_temporal_blocks(df_trainval)

df_trainval["spatial_block"] = df_trainval["profile_id"].apply(spatial_fn)
df_trainval["temporal_block"] = df_trainval["profile_id"].apply(temporal_fn)

# Sanity Check - Each profile must belong to exactly one block
assert df_trainval.groupby("profile_id")["spatial_block"].nunique().max() == 1
assert df_trainval.groupby("profile_id")["temporal_block"].nunique().max() == 1

# 5 — CV Fold Definition

In [7]:
def make_cv_folds(df, min_profiles=10):
    folds = []
    for s in sorted(df.spatial_block.unique()):
        for t in sorted(df.temporal_block.unique()):
            val_df = df[(df.spatial_block==s) & (df.temporal_block==t)]
            if val_df["profile_id"].nunique() >= min_profiles:
                folds.append((s, t))
    return folds
folds = make_cv_folds(df_trainval)
print("Total CV folds:", len(folds))

Total CV folds: 37


# 6 — Dataset & Collate (Padding + Masking)

In [8]:
class ProfileDataset(Dataset):
    def __init__(self, df, feature_cols, target_col):
        self.data = []
        for _, g in df.groupby("profile_id"):
            self.data.append({
                "X": g[feature_cols].values.astype(np.float32),
                "depth": g["Depth(m)"].values.astype(np.float32),
                "y": g[target_col].values.astype(np.float32),
                "L": len(g)
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def collate_profiles(batch):
    B = len(batch)
    max_L = max(b["L"] for b in batch)
    F = batch[0]["X"].shape[1]

    X = torch.zeros(B, max_L, F)
    depth = torch.zeros(B, max_L)
    y = torch.zeros(B, max_L)
    mask = torch.ones(B, max_L, dtype=torch.bool)

    for i, b in enumerate(batch):
        L = b["L"]
        X[i, :L] = torch.from_numpy(b["X"])
        depth[i, :L] = torch.from_numpy(b["depth"])
        y[i, :L] = torch.from_numpy(b["y"])
        mask[i, :L] = False

    return X, depth, y, mask

# 7 — Transformer Model (Depth Sinusoidal PE)

In [9]:
class DepthPositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model

        div = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        self.register_buffer("div", div)

    def forward(self, depth):
        depth = depth.unsqueeze(-1)
        pe = torch.zeros(
            depth.size(0), depth.size(1), self.d_model,
            device=depth.device
        )
        pe[..., 0::2] = torch.sin(depth * self.div)
        pe[..., 1::2] = torch.cos(depth * self.div)
        return pe


class SalinityTransformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=8, layers=4):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        self.pe = DepthPositionalEncoding(d_model)

        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc, layers)
        self.out = nn.Linear(d_model, 1)

    def forward(self, x, depth, mask):
        assert depth.dim() == 2, "Depth must be (B, L)"

        x = self.proj(x) + self.pe(depth)
        x = self.encoder(x, src_key_padding_mask=mask)
        return self.out(x).squeeze(-1)

# 8 — Losses & Early Stopping

In [10]:
def masked_mse(pred, y, mask):
    return ((pred - y) ** 2)[~mask].mean()


def masked_rmse(pred, y, mask):
    return torch.sqrt(masked_mse(pred, y, mask))


class EarlyStopping:
    def __init__(self, patience=12):
        self.best = float("inf")
        self.count = 0
        self.patience = patience
        self.state = None

    def step(self, metric, model):
        if metric < self.best:
            self.best = metric
            self.count = 0
            self.state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False
        self.count += 1
        return self.count >= self.patience

# 9 — Nested CV Training

In [ ]:
# ============================================================
# CELL — SPATIO-TEMPORAL CV FOR HYPERPARAMETER TUNING
# ============================================================

HYPERPARAM_GRID = [
    {"d_model": 64,  "nhead": 4, "layers": 2, "lr": 1e-4},
    {"d_model": 128, "nhead": 8, "layers": 4, "lr": 1e-4},
    {"d_model": 128, "nhead": 8, "layers": 4, "lr": 3e-4},
]

cv_results = []

for hp_id, hp in enumerate(HYPERPARAM_GRID):
    print("\n" + "=" * 60)
    print(f"Hyperparameter set {hp_id}")
    print(hp)
    print("=" * 60)

    fold_scores = []

    # --------------------------------------------------------
    # LOOP OVER SPATIO-TEMPORAL FOLDS
    # --------------------------------------------------------
    for fold_id, (s, t) in enumerate(folds):
        print(f"\n--- Fold {fold_id} | Spatial {s} | Temporal {t} ---")

        val_mask = (
            (df_trainval.spatial_block == s) &
            (df_trainval.temporal_block == t)
        )

        train_df = df_trainval[~val_mask].copy()
        val_df   = df_trainval[val_mask].copy()

        if val_df["profile_id"].nunique() < 10:
            print("Skipping fold (too few profiles)")
            continue

        # ----------------------------
        # SCALING (TRAIN ONLY)
        # ----------------------------
        scaler = StandardScaler()
        train_df[FEATURE_COLS] = scaler.fit_transform(train_df[FEATURE_COLS])
        val_df[FEATURE_COLS]   = scaler.transform(val_df[FEATURE_COLS])

        # ----------------------------
        # DATA LOADERS
        # ----------------------------
        train_loader = DataLoader(
            ProfileDataset(train_df, FEATURE_COLS, TARGET_COL),
            batch_size=8,
            shuffle=True,
            collate_fn=collate_profiles
        )

        val_loader = DataLoader(
            ProfileDataset(val_df, FEATURE_COLS, TARGET_COL),
            batch_size=8,
            shuffle=False,
            collate_fn=collate_profiles
        )

        # ----------------------------
        # MODEL
        # ----------------------------
        model = SalinityTransformer(
            input_dim=len(FEATURE_COLS),
            d_model=hp["d_model"],
            nhead=hp["nhead"],
            layers=hp["layers"]
        ).to(DEVICE)

        optimizer = torch.optim.AdamW(
            model.parameters(), lr=hp["lr"]
        )

        stopper = EarlyStopping(patience=10)

        # ----------------------------
        # TRAINING LOOP
        # ----------------------------
        for epoch in range(100):
            model.train()

            for X, d, y, m in train_loader:
                X, d, y, m = (
                    X.to(DEVICE),
                    d.to(DEVICE),
                    y.to(DEVICE),
                    m.to(DEVICE)
                )

                loss = masked_mse(model(X, d, m), y, m)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # ------------------------
            # VALIDATION
            # ------------------------
            model.eval()
            rmses = []

            with torch.no_grad():
                for X, d, y, m in val_loader:
                    X, d, y, m = (
                        X.to(DEVICE),
                        d.to(DEVICE),
                        y.to(DEVICE),
                        m.to(DEVICE)
                    )

                    rmses.append(
                        masked_rmse(model(X, d, m), y, m).item()
                    )

            val_rmse = float(np.mean(rmses))
            print(f"Epoch {epoch:03d} | Val RMSE: {val_rmse:.4f}")

            if stopper.step(val_rmse, model):
                print("Early stopping")
                break

        # ----------------------------
        # STORE BEST SCORE FOR THIS FOLD
        # ----------------------------
        fold_scores.append(stopper.best)

    # --------------------------------------------------------
    # MEAN CV SCORE FOR THIS HYPERPARAM SET
    # --------------------------------------------------------
    mean_rmse = float(np.mean(fold_scores))

    cv_results.append({
        "hyperparams": hp,
        "mean_rmse": mean_rmse
    })

    print(f"\n>>> Mean CV RMSE: {mean_rmse:.4f}")

# ============================================================
# SELECT BEST HYPERPARAMETERS (AFTER CV)
# ============================================================

best_run = min(cv_results, key=lambda x: x["mean_rmse"])
BEST_HP = best_run["hyperparams"]

print("\n BEST HYPERPARAMETERS (CV-selected):")
print(BEST_HP)


Hyperparameter set 0
{'d_model': 64, 'nhead': 4, 'layers': 2, 'lr': 0.0001}

--- Fold 0 | Spatial 0 | Temporal 0 ---


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 000 | Val RMSE: 29.1867
Epoch 001 | Val RMSE: 27.3446
Epoch 002 | Val RMSE: 25.2669
Epoch 003 | Val RMSE: 23.0041
Epoch 004 | Val RMSE: 20.6292
Epoch 005 | Val RMSE: 18.2015
Epoch 006 | Val RMSE: 15.7726
Epoch 007 | Val RMSE: 13.3932
Epoch 008 | Val RMSE: 11.1133
Epoch 009 | Val RMSE: 8.9808
Epoch 010 | Val RMSE: 7.0409
Epoch 011 | Val RMSE: 5.3330
Epoch 012 | Val RMSE: 3.8880
Epoch 013 | Val RMSE: 2.7249
Epoch 014 | Val RMSE: 1.8428
Epoch 015 | Val RMSE: 1.2206
Epoch 016 | Val RMSE: 0.8217
Epoch 017 | Val RMSE: 0.5991
Epoch 018 | Val RMSE: 0.4920
Epoch 019 | Val RMSE: 0.4471
Epoch 020 | Val RMSE: 0.4286
Epoch 021 | Val RMSE: 0.4070
Epoch 022 | Val RMSE: 0.3910
Epoch 023 | Val RMSE: 0.3821
Epoch 024 | Val RMSE: 0.3740
Epoch 025 | Val RMSE: 0.3691
Epoch 026 | Val RMSE: 0.3706
Epoch 027 | Val RMSE: 0.3489
Epoch 028 | Val RMSE: 0.3370
Epoch 029 | Val RMSE: 0.3225
Epoch 030 | Val RMSE: 0.3203
Epoch 031 | Val RMSE: 0.2977
Epoch 032 | Val RMSE: 0.2894
Epoch 033 | Val RMSE: 0.2692
Epoch

# 10 - Final Training Loop

In [12]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

final_scaler = StandardScaler()

# Split profiles FIRST
profiles = df_trainval["profile_id"].unique()
np.random.shuffle(profiles)

val_profiles   = profiles[:int(0.1 * len(profiles))]
train_profiles = profiles[int(0.1 * len(profiles)):]

train_df = df_trainval[df_trainval.profile_id.isin(train_profiles)].copy()
val_df   = df_trainval[df_trainval.profile_id.isin(val_profiles)].copy()

# Scale AFTER split
final_scaler = StandardScaler()
train_df[FEATURE_COLS] = final_scaler.fit_transform(train_df[FEATURE_COLS])
val_df[FEATURE_COLS]   = final_scaler.transform(val_df[FEATURE_COLS])

train_loader = DataLoader(
    ProfileDataset(train_df, FEATURE_COLS, TARGET_COL),
    batch_size=8, shuffle=True,
    collate_fn=collate_profiles
)

val_loader = DataLoader(
    ProfileDataset(val_df, FEATURE_COLS, TARGET_COL),
    batch_size=8, shuffle=False,
    collate_fn=collate_profiles
)

final_model = SalinityTransformer(
    input_dim=len(FEATURE_COLS),
    d_model=BEST_HP["d_model"],
    nhead=BEST_HP["nhead"],
    layers=BEST_HP["layers"]
).to(DEVICE)

optimizer = torch.optim.AdamW(
    final_model.parameters(), lr=BEST_HP["lr"]
)
stopper = EarlyStopping(patience=10)

for epoch in range(100):
    final_model.train()
    for X, d, y, m in train_loader:
        X, d, y, m = X.to(DEVICE), d.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
        loss = masked_mse(final_model(X, d, m), y, m)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    final_model.eval()
    rmses = []
    with torch.no_grad():
        for X, d, y, m in val_loader:
            X, d, y, m = X.to(DEVICE), d.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
            rmses.append(masked_rmse(final_model(X, d, m), y, m).item())

    val_rmse = float(np.mean(rmses))
    print(f"[FINAL] Epoch {epoch:03d} | Val RMSE: {val_rmse:.4f}")

    if stopper.step(val_rmse, final_model):
        print("Final early stopping")
        break

final_model.load_state_dict(stopper.state)
print(f"Restored best model from early stopping | Best Val RMSE: {stopper.best:.4f}")

# ==============================
# FINAL CHECKPOINT SAVING
# ==============================

final_ckpt_path = "/kaggle/working/final_transformer.pt"

# ---- Model architecture config ----
model_config = {
    "input_dim": len(FEATURE_COLS),
    "d_model": BEST_HP["d_model"],
    "nhead": BEST_HP["nhead"],
    "layers": BEST_HP["layers"]
}

# ---- Data / feature config ----
data_config = {
    "feature_cols": FEATURE_COLS,
    "target_col": TARGET_COL,
    "train_profiles": train_profiles,
    "val_profiles": val_profiles
}

# ---- Training metadata ----
training_meta = {
    "best_val_rmse": stopper.best,
    "epochs_trained": epoch + 1,
    "batch_size": 8,
    "optimizer": "AdamW",
    "learning_rate": BEST_HP["lr"],
    "loss": "masked_mse",
    "early_stopping_patience": stopper.patience,
    "device": DEVICE
}

# ---- Unified checkpoint ----
checkpoint = {
    "model_state_dict": final_model.state_dict(),
    "model_config": model_config,
    "data_config": data_config,
    "training_meta": training_meta
}

# ---- Save everything ----
torch.save(checkpoint, final_ckpt_path)

# ---- Save scaler separately (best practice) ----
joblib.dump(
    final_scaler,
    "/kaggle/working/final_transformer_scaler.joblib"
)

print("Final model, config, metadata, and scaler saved successfully.")


[FINAL] Epoch 000 | Val RMSE: 29.1912
[FINAL] Epoch 001 | Val RMSE: 27.5570
[FINAL] Epoch 002 | Val RMSE: 25.7268
[FINAL] Epoch 003 | Val RMSE: 23.7346
[FINAL] Epoch 004 | Val RMSE: 21.6147
[FINAL] Epoch 005 | Val RMSE: 19.4277
[FINAL] Epoch 006 | Val RMSE: 17.2216
[FINAL] Epoch 007 | Val RMSE: 15.0323
[FINAL] Epoch 008 | Val RMSE: 12.8903
[FINAL] Epoch 009 | Val RMSE: 10.8300
[FINAL] Epoch 010 | Val RMSE: 8.8898
[FINAL] Epoch 011 | Val RMSE: 7.1058
[FINAL] Epoch 012 | Val RMSE: 5.5093
[FINAL] Epoch 013 | Val RMSE: 4.1287
[FINAL] Epoch 014 | Val RMSE: 2.9787
[FINAL] Epoch 015 | Val RMSE: 2.0641
[FINAL] Epoch 016 | Val RMSE: 1.3753
[FINAL] Epoch 017 | Val RMSE: 0.8928
[FINAL] Epoch 018 | Val RMSE: 0.5913
[FINAL] Epoch 019 | Val RMSE: 0.4324
[FINAL] Epoch 020 | Val RMSE: 0.3722
[FINAL] Epoch 021 | Val RMSE: 0.3563
[FINAL] Epoch 022 | Val RMSE: 0.3528
[FINAL] Epoch 023 | Val RMSE: 0.3333
[FINAL] Epoch 024 | Val RMSE: 0.3253
[FINAL] Epoch 025 | Val RMSE: 0.3199
[FINAL] Epoch 026 | Val RMSE

# 11 — Final Test Evaluation

In [13]:
# -----------------------------------------
# APPLY FINAL SCALER TO TEST SET (NO LEAKAGE)
# -----------------------------------------
test_df = df_test.copy()

test_df.loc[:, FEATURE_COLS] = final_scaler.transform(
    test_df[FEATURE_COLS]
)

test_loader = DataLoader(
    ProfileDataset(test_df, FEATURE_COLS, TARGET_COL),
    batch_size=8,
    shuffle=False,
    collate_fn=collate_profiles
)

final_model.eval()

all_pred  = []
all_true  = []

with torch.no_grad():
    for X, depth, y, mask in test_loader:
        X     = X.to(DEVICE)
        depth = depth.to(DEVICE)
        y     = y.to(DEVICE)
        mask  = mask.to(DEVICE)

        pred = final_model(X, depth, mask)

        valid = ~mask  # valid (non-padded) points

        all_pred.append(pred[valid].cpu().numpy())
        all_true.append(y[valid].cpu().numpy())

all_pred = np.concatenate(all_pred)
all_true = np.concatenate(all_true)

# -----------------------------------------
# TEST METRICS
# -----------------------------------------
rmse = np.sqrt(np.mean((all_pred - all_true) ** 2))
mae  = np.mean(np.abs(all_pred - all_true))

ss_res = np.sum((all_true - all_pred) ** 2)
ss_tot = np.sum((all_true - np.mean(all_true)) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"TEST RMSE : {rmse:.4f}")
print(f"TEST MAE  : {mae:.4f}")
print(f"TEST R²   : {r2:.4f}")

TEST RMSE : 0.1409
TEST MAE  : 0.0686
TEST R²   : 0.8098


# 13 — Sanity Checks

In [14]:
import torch
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

def sanity_plot_random_profile_transformer(
    raw_data_path,       # Path to raw CSV/Parquet
    model_ckpt_path,     # Path to saved Transformer checkpoint (.pt)
    scaler_path,         # Path to saved scaler (.joblib)
    feature_cols,        # List of feature columns
    target_col= "salinity", # Target column name
    device="cuda"        # "cuda" or "cpu"
):
    # -----------------------------
    # 1️⃣ Load raw data
    # -----------------------------
    if raw_data_path.endswith(".parquet"):
        df = pd.read_parquet(raw_data_path)
    else:
        df = pd.read_csv(raw_data_path)
    
    # -----------------------------
    # 2️⃣ Load scaler
    # -----------------------------
    scaler = joblib.load(scaler_path)

    # -----------------------------
    # 3️⃣ Load model checkpoint
    # -----------------------------
    checkpoint = torch.load(model_ckpt_path, map_location=device)
    model_config = checkpoint['model_config']

    # Make sure SalinityTransformer class is defined in this cell or imported
    model = SalinityTransformer(
        input_dim=model_config['input_dim'],
        d_model=model_config['d_model'],
        nhead=model_config['nhead'],
        layers=model_config['layers']
    ).to(device)

    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # -----------------------------
    # 4️⃣ Pick one random profile
    # -----------------------------
    profile_id = np.random.choice(df["profile_id"].unique())
    df_p = df[df["profile_id"] == profile_id].copy()

    # -----------------------------
    # 5️⃣ Scale features
    # -----------------------------
    df_p.loc[:, feature_cols] = scaler.transform(df_p[feature_cols])

    # -----------------------------
    # 6️⃣ Dataset + loader
    # -----------------------------
    loader = DataLoader(
        ProfileDataset(df_p, feature_cols, target_col),
        batch_size=1,
        shuffle=False,
        collate_fn=collate_profiles
    )

    # -----------------------------
    # 7️⃣ Forward pass
    # -----------------------------
    with torch.no_grad():
        X, depth, y, mask = next(iter(loader))
        X = X.to(device)
        depth = depth.to(device)
        y = y.to(device)
        mask = mask.to(device)

        pred = model(X, depth, mask)

        valid = ~mask
        depth_vals = depth[valid].cpu().numpy()
        y_true = y[valid].cpu().numpy()
        y_pred = pred[valid].cpu().numpy()

    # -----------------------------
    # 8️⃣ Plot
    # -----------------------------
    plt.figure(figsize=(5, 7))
    plt.plot(y_true, depth_vals, "o-", label="Observed")
    plt.plot(y_pred, depth_vals, "x--", label="Predicted")
    plt.gca().invert_yaxis()
    plt.xlabel("Salinity (PSU)")
    plt.ylabel("Depth (m)")
    plt.title(f"Transformer sanity check | Profile {profile_id}")
    plt.legend()
    plt.grid(True)
    plt.show()

sanity_plot_random_profile_transformer(
    raw_data_path="/kaggle/input/transformer",
    model_ckpt_path="/kaggle/working/final_transformer.pt",
    scaler_path="/kaggle/working/final_transformer_scaler.joblib",
    device="cuda"
)

TypeError: sanity_plot_random_profile_transformer() missing 1 required positional argument: 'feature_cols'